# 03 — Model Öyrətmə və Submission

Apar — İstifadəçi Rəylərinin Təsnifatı

In [ ]:
import pandas as pd
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, f1_score
import sys, random

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

train = pd.read_csv('../data/train_processed.csv')
test  = pd.read_csv('../data/test_processed.csv')

## Baseline Pipeline (TF-IDF + Logistic Regression)

In [ ]:
X = train['text'].fillna('')
y = train['label']

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        analyzer='char_wb',
        ngram_range=(2, 4),
        max_features=100_000,
        sublinear_tf=True
    )),
    ('clf', LogisticRegression(
        max_iter=1000,
        C=1.0,
        class_weight='balanced',
        random_state=SEED
    ))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
scores = cross_val_score(pipeline, X, y, cv=cv, scoring='f1_macro', n_jobs=-1)
print(f'CV Macro F1: {scores.mean():.4f} ± {scores.std():.4f}')

## Modeli Tam Öyrət və Submission Yarat

In [ ]:
pipeline.fit(X, y)

X_test = test['text'].fillna('')
preds  = pipeline.predict(X_test)

submission = pd.DataFrame({'id': test['id'], 'label': preds})
submission.to_csv('../outputs/submission.csv', index=False)
print('Submission hazırdır!')
submission.head(10)

## Label Paylanması (Submission)

In [ ]:
submission['label'].value_counts()